# CrewAI and Other Multi-Agent Frameworks, Hands-On

> **Who is this for:** learners who have seen the short [multi-agent frameworks overview](04_multi_agent_frameworks.ipynb) and want to understand, with running code, how a "crew" of agents is really put together, how the popular frameworks differ, and when multi-agent is (and is not) a good idea.
>
> **What you will learn:**
> - the CrewAI mental model: **Agent** (role, goal, backstory, tools, llm), **Task** (description, expected_output, agent, context), **Crew** (agents, tasks, **process**: sequential or hierarchical with a manager)
> - how CrewAI turns those objects into prompts and passes **context** from task to task, by building a tiny **mini-crew engine** in pure Python
> - real CrewAI code run **offline** with a scripted fake LLM: tools, structured (Pydantic) output, guardrails, the hierarchical manager, `kickoff` / `kickoff_async` / `kickoff_for_each`
> - **Flows**: event-driven workflows with `@start`, `@listen` and `@router`, and typed state
> - memory, knowledge, and the YAML project layout from `crewai create crew`
> - the same **research -> write -> review** pipeline in LangGraph (runnable), CrewAI, AutoGen / Microsoft Agent Framework, OpenAI Agents SDK, Google ADK, PydanticAI, smolagents and LlamaIndex Workflows
> - a comparison table, the main **failure modes** of multi-agent systems (coordination loops, cost blow-ups, error cascades) with small demos, and **when not to use multi-agent at all**
>
> **Prerequisites:** [Agent fundamentals](01_agent_fundamentals.ipynb), [Multi-agent frameworks overview](04_multi_agent_frameworks.ipynb), and ideally [LangGraph hands-on](12_langgraph_hands_on.ipynb). Related: [Advanced agent frameworks](07_advanced_agent_frameworks.ipynb), [Harness engineering and agentic engineering](10_harness_engineering_and_agentic_engineering.ipynb), [Context engineering, memory and agent protocols](11_context_engineering_memory_and_agent_protocols.ipynb).
>
> **Time needed:** about 2 hours to read and play (the code runs in under a minute; importing CrewAI alone can take several seconds).
>
> **Runs on:** CPU, no API key, no internet. Every "LLM" here is a **scripted stand-in** (a Python function with prepared answers), and we say so each time. The CrewAI cells run only if `crewai` is installed; otherwise they print `skipped: pip install crewai` and the pure-Python mini-crew still shows every idea. Tested with crewai 1.15 and langgraph 1.x (as of late 2026). Code snippets for the other frameworks are shown as text and kept **version-agnostic**: their APIs change often, so check each project's docs before copying.

## 0. Setup

We switch off CrewAI's anonymous telemetry and tracing (they would try to use the internet), point its local storage at a temporary folder, and then **try** to import CrewAI. `HAVE_CREWAI` tells later cells whether to run the real CrewAI code.

In [1]:
import os, re, json, random, asyncio, tempfile, operator, textwrap, warnings
from dataclasses import dataclass, field
from typing import Annotated, Callable, Literal
from typing_extensions import TypedDict

random.seed(42)
warnings.filterwarnings("ignore")

# CrewAI settings: no telemetry, no cloud tracing, local files go to a temp folder
os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")
os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")
os.environ.setdefault("CREWAI_STORAGE_DIR", tempfile.mkdtemp(prefix="crewai_demo_"))

try:
    import crewai
    from crewai import Agent, Task, Crew, Process, BaseLLM
    HAVE_CREWAI = True
    print("crewai", crewai.__version__, "found: the real CrewAI cells will run (offline, scripted LLM).")
except Exception as e:          # not installed (or broken install)
    HAVE_CREWAI = False
    print("skipped: pip install crewai   (the pure-Python mini-crew cells still run)")

from importlib.metadata import version
print("langgraph", version("langgraph"))

crewai 1.15.41 found: the real CrewAI cells will run (offline, scripted LLM).
langgraph 1.2.21


## 1. The CrewAI mental model

**What it is.** CrewAI is an open-source Python framework for **role-based** multi-agent systems. You describe a small team the way you would describe a team of people: who is on it, what each person is good at, which jobs need doing, and in what order.

**Simple analogy.** A film crew. Each member has a **role** (director, camera operator, editor), a **goal**, and **experience** (backstory). The script lists **tasks** (shoot scene 1, edit scene 1). The **process** decides the order: follow the script in order (sequential), or let the director hand out jobs (hierarchical).

| Concept | What it is (one sentence) | Key fields |
|---|---|---|
| **Agent** | an LLM with a persona and tools | `role`, `goal`, `backstory`, `tools`, `llm`, `allow_delegation`, `max_iter` |
| **Task** | one job with a clear "done" description | `description`, `expected_output`, `agent`, `context`, `output_pydantic`, `guardrail` |
| **Crew** | the team plus the order of work | `agents`, `tasks`, `process`, `manager_llm` / `manager_agent`, `memory` |
| **Process** | how tasks are scheduled | `Process.sequential`, `Process.hierarchical` |
| **Tool** | a function an agent may call | `@tool` decorator or a `BaseTool` subclass |
| **Flow** | an event-driven workflow around crews and plain code | `@start`, `@listen`, `@router`, `self.state` |
| **Memory / Knowledge** | remembered facts / documents the agents can search | `memory=True`, `knowledge_sources=[...]` |
| **Guardrail** | a check on a task's output, with automatic retries | a function `(output) -> (ok, value_or_feedback)` |

```mermaid
flowchart LR
    I[inputs: topic] --> T1
    subgraph Crew [Crew: process = sequential]
      T1[Task 1: research<br/>agent = Researcher] -- context --> T2[Task 2: write<br/>agent = Writer]
      T2 -- context --> T3[Task 3: review<br/>agent = Reviewer]
    end
    T3 --> O[CrewOutput]
```

**When to use CrewAI.** When your problem is naturally "a few roles doing a known sequence of jobs" and you want to get there with little code. When you need fine-grained control over every step, loops and state, a graph framework like [LangGraph](12_langgraph_hands_on.ipynb) usually fits better (section 6 compares them).

## 2. Build a mini-crew engine in pure Python

Before using the real library, we build a **tiny copy** of its core ideas in about 80 lines. This shows that there is no magic: a "crew" is mostly **prompt building + passing outputs along**.

**How it works.**
1. An **agent** turns its role/goal/backstory into a **system prompt** (the instructions an LLM reads first).
2. A **task** becomes the **user prompt**: description + expected output + **context** (outputs of earlier tasks).
3. The agent calls its LLM. If the LLM asks for a tool (we use the text format `ACTION: tool_name | input`), the agent runs the tool, appends the result, and asks again, up to `max_iter` times.
4. The **crew** runs the tasks: in order (**sequential**), or by asking a **manager** which agent should do each task (**hierarchical**).

The LLM is a **scripted stand-in**: a Python function that looks at the prompt and returns a prepared answer.

In [2]:
@dataclass
class MiniAgent:
    role: str
    goal: str
    backstory: str
    llm: Callable[[str, str], str]           # (system_prompt, user_prompt) -> text
    tools: dict = field(default_factory=dict)  # name -> python function
    max_iter: int = 3                          # safety limit on tool loops

    def system_prompt(self) -> str:
        tools = ", ".join(self.tools) or "none"
        return f"You are {self.role}. {self.backstory}\nYour goal: {self.goal}\nTools: {tools}"

    def execute(self, task: "MiniTask", context: str) -> str:
        user = f"Task: {task.description}\nExpected output: {task.expected_output}"
        if context:
            user += f"\nContext from earlier tasks:\n{context}"
        for _ in range(self.max_iter):
            reply = self.llm(self.system_prompt(), user)
            if reply.startswith("ACTION:"):                 # the LLM wants a tool
                name, arg = [x.strip() for x in reply[len("ACTION:"):].split("|", 1)]
                result = self.tools[name](arg)
                user += f"\nObservation from {name}: {result}"
                continue
            return reply                                    # a final answer
        return "Stopped: too many tool calls"

@dataclass
class MiniTask:
    description: str
    expected_output: str
    agent: MiniAgent | None = None
    context: list | None = None     # None = "all previous outputs" (CrewAI's default in sequential)
    output: str = ""

@dataclass
class MiniCrew:
    agents: list
    tasks: list
    process: str = "sequential"      # or "hierarchical"
    manager_llm: Callable | None = None
    verbose: bool = True

    def kickoff(self, inputs: dict) -> str:
        done = []
        for task in self.tasks:
            task.description = task.description.format(**inputs)    # fill {placeholders}
            if self.process == "hierarchical":                       # the manager picks the agent
                roles = [a.role for a in self.agents]
                pick = self.manager_llm(f"Who should do: {task.description}? Options: {roles}", "")
                agent = next(a for a in self.agents if a.role == pick)
            else:
                agent = task.agent
            ctx_tasks = done if task.context is None else task.context
            context = "\n".join(f"[{t.agent.role}] {t.output}" for t in ctx_tasks)
            task.agent = agent
            task.output = agent.execute(task, context)
            done.append(task)
            if self.verbose:
                print(f"  {agent.role:<10} | {task.description[:40]:<40} -> {task.output[:70]}")
        return done[-1].output

In [3]:
# --- a scripted stand-in LLM: picks an answer based on who is asking and what is in the prompt
def fake_llm(system: str, user: str) -> str:
    if "You are Researcher" in system:
        if "Observation" not in user:
            return "ACTION: search | LangGraph"                    # step 1: ask for a tool
        return "Facts: (1) it is open source (2) it is written in Python (3) it supports tools"
    if "You are Writer" in system:
        topic = re.search(r"about (\w+)", user)
        facts = re.search(r"Facts: (.*)", user)
        return f"Draft: {topic.group(1) if topic else '?'} is a library for agents; {facts.group(1) if facts else '?'}."
    if "You are Reviewer" in system:
        return "APPROVED (clear and based on the facts)" if "Draft:" in user else "REJECTED: no draft"
    return "?"

def fake_search(query: str) -> str:           # a toy tool (no internet)
    return f"3 results about {query}"

researcher = MiniAgent("Researcher", "find correct facts", "You are careful and cite sources.",
                       fake_llm, tools={"search": fake_search})
writer     = MiniAgent("Writer", "write clear text", "You write for beginners.", fake_llm)
reviewer   = MiniAgent("Reviewer", "check quality", "You are strict but fair.", fake_llm)

tasks = [MiniTask("Research {topic}", "3 short facts", researcher),
         MiniTask("Write one paragraph about {topic}", "one paragraph", writer),
         MiniTask("Review the paragraph", "APPROVED or REJECTED + reason", reviewer)]

print("System prompt built for the researcher:\n" + textwrap.indent(researcher.system_prompt(), "    "))
print("\nSequential run:")
final = MiniCrew([researcher, writer, reviewer], tasks).kickoff({"topic": "LangGraph"})
print("\nFinal output:", final)

System prompt built for the researcher:
    You are Researcher. You are careful and cite sources.
    Your goal: find correct facts
    Tools: search

Sequential run:
  Researcher | Research LangGraph                       -> Facts: (1) it is open source (2) it is written in Python (3) it suppor
  Writer     | Write one paragraph about LangGraph      -> Draft: LangGraph is a library for agents; (1) it is open source (2) it
  Reviewer   | Review the paragraph                     -> APPROVED (clear and based on the facts)

Final output: APPROVED (clear and based on the facts)


**What just happened?** Each task's output became **context** for the next task, because we did not set `context` explicitly (this mirrors CrewAI's sequential default: a task sees all earlier outputs). The researcher first asked for a tool, got an "Observation", then answered.

Now the **hierarchical** process: there is no fixed `agent` on the tasks. A **manager** (another LLM) decides who does each job. In real CrewAI the manager also delegates with tools and can ask follow-up questions (section 3.6).

In [4]:
def fake_manager(question: str, _: str) -> str:
    # Scripted manager: choose a worker by keywords in the job description
    q = question.lower().split("options")[0]      # only read the job, not the list of names
    return "Researcher" if "research" in q else "Reviewer" if "review" in q else "Writer"

h_tasks = [MiniTask("Research {topic}", "3 facts"),           # note: no agent assigned
           MiniTask("Write one paragraph about {topic}", "one paragraph"),
           MiniTask("Review the paragraph", "verdict")]
print("Hierarchical run (manager assigns agents):")
MiniCrew([researcher, writer, reviewer], h_tasks, process="hierarchical",
         manager_llm=fake_manager).kickoff({"topic": "CrewAI"})

Hierarchical run (manager assigns agents):
  Researcher | Research CrewAI                          -> Facts: (1) it is open source (2) it is written in Python (3) it suppor
  Writer     | Write one paragraph about CrewAI         -> Draft: CrewAI is a library for agents; (1) it is open source (2) it is
  Reviewer   | Review the paragraph                     -> APPROVED (clear and based on the facts)


'APPROVED (clear and based on the facts)'

### 2.1 A mini Flow: `@start`, `@listen`, `@router`

**What it is.** A **Flow** is an **event-driven** workflow: methods run when something they "listen" to has finished. `@start` marks the first method(s), `@listen(x)` runs after method `x` (or after a router emits the label `x`), and `@router(x)` runs after `x` and returns a **label** that picks the next branch.

**Analogy.** Dominoes with switches: when one falls it knocks over the ones listening to it, and a router is a switch that decides which line of dominoes falls next.

Here is a 30-line engine that mimics this behaviour (CrewAI's real one adds async, persistence, typed state and more).

In [5]:
def start():
    def deco(fn): fn._start = True; return fn
    return deco
def listen(*triggers):            # run when ANY of the triggers fires (CrewAI writes this as or_(...))
    def deco(fn):
        fn._listen = {t if isinstance(t, str) else t.__name__ for t in triggers}; return fn
    return deco
def router(trigger):
    def deco(fn): fn._router = trigger.__name__; return fn
    return deco

class MiniFlow:
    def __init__(self):
        self.state = {}
    def kickoff(self, max_steps=20):
        methods = [getattr(self, n) for n in dir(self) if callable(getattr(self, n)) and not n.startswith("_")]
        queue = [m.__name__ for m in methods if getattr(m, "_start", False)]
        result = None
        for m in methods:                      # run the start methods first
            if getattr(m, "_start", False):
                result = m()
        steps = 0
        while queue and steps < max_steps:     # an event = a finished method name or a router label
            event = queue.pop(0); steps += 1
            for m in methods:
                if event in getattr(m, "_listen", set()):
                    result = m(); queue.append(m.__name__)
                if getattr(m, "_router", None) == event:
                    label = m(); print(f"    router {m.__name__!r} -> {label!r}"); queue.append(label)
        return result

class BlogFlow(MiniFlow):
    @start()
    def pick_topic(self):
        self.state.update(topic="agents", tries=0)
    @listen(pick_topic, "retry")       # runs after pick_topic, and again on every "retry"
    def write_draft(self):
        self.state["tries"] += 1
        self.state["draft"] = f"draft v{self.state['tries']} about {self.state['topic']}"
    @router(write_draft)
    def check_quality(self):
        return "good" if self.state["tries"] >= 2 else "retry"     # toy rule: 2nd draft is good
    @listen("good")
    def publish(self):
        return "published: " + self.state["draft"]

flow = BlogFlow()
print(flow.kickoff())
print("final state:", flow.state)

    router 'check_quality' -> 'retry'
    router 'check_quality' -> 'good'
published: draft v2 about agents
final state: {'topic': 'agents', 'tries': 2, 'draft': 'draft v2 about agents'}


## 3. Real CrewAI, offline, with a scripted LLM

**Can CrewAI run without an API key?** Yes. CrewAI lets you plug in your **own LLM class** by subclassing `BaseLLM` and implementing `call(messages, ...)`, which must return the model's text. Our `ScriptedLLM` returns prepared answers chosen by the **role of the agent that is calling**, so the whole crew runs offline. (In real use you would write `llm="provider/model-name"` or `LLM(model=...)` and set an API key.)

**How CrewAI talks to a text-only LLM.** When an LLM does not support native function calling, CrewAI uses a **ReAct-style text format** in its prompt: the model answers either

```
Thought: ...
Action: tool_name
Action Input: {"arg": "value"}
```
to call a tool, or
```
Thought: ...
Final Answer: the result
```
to finish. Our scripted replies follow that format.

**One Jupyter detail.** Jupyter itself runs an `asyncio` **event loop** (the engine that runs async code). Recent CrewAI versions refuse to run a *synchronous* `crew.kickoff()` inside a running loop, so in this notebook we call a tiny helper `kickoff(crew, ...)` that runs `crew.kickoff(...)` in a separate thread. In a normal `.py` script you call `crew.kickoff(...)` directly, and in async code you `await crew.kickoff_async(...)`.

In [6]:
if HAVE_CREWAI:
    class ScriptedLLM(BaseLLM):
        """FAKE LLM for offline demos. `replies` maps an agent role to a function
        (messages, call_number) -> text. Every call is recorded in `log`."""
        replies: dict = {}
        log: list = []
        counts: dict = {}

        def call(self, messages, tools=None, callbacks=None, available_functions=None,
                 from_task=None, from_agent=None, response_model=None, **kwargs):
            role = from_agent.role if from_agent is not None else "unknown"
            self.counts[role] = self.counts.get(role, 0) + 1
            self.log.append({"role": role, "messages": messages})
            reply_fn = self.replies.get(role, lambda m, n: "Thought: done\nFinal Answer: (no script)")
            return reply_fn(messages, self.counts[role])

        def supports_function_calling(self) -> bool:
            return False                       # use the text (ReAct) format

        def get_input_tokens_count(self, messages, tools=None, response_model=None) -> int:
            return len(str(messages)) // 4      # rough estimate, avoids a warning

    final = lambda text: f"Thought: I can answer now.\nFinal Answer: {text}"   # helper

    from concurrent.futures import ThreadPoolExecutor
    def kickoff(crew, **kwargs):
        """Run crew.kickoff(...) in a helper thread. Jupyter already runs an asyncio event loop,
        and recent CrewAI versions refuse a synchronous kickoff inside one.
        In a normal Python script just call crew.kickoff(...) directly."""
        with ThreadPoolExecutor(1) as pool:
            return pool.submit(crew.kickoff, **kwargs).result()
    print("ScriptedLLM and kickoff() helper ready")
else:
    print("skipped: pip install crewai")

ScriptedLLM and kickoff() helper ready


### 3.1 Agents and Tasks are just configuration objects

Creating an `Agent` or a `Task` does **not** call the LLM. They are Pydantic models (validated data objects) that hold configuration. Writing `{topic}` in a description makes it a **placeholder** that is filled from `kickoff(inputs={...})`.

In [7]:
if HAVE_CREWAI:
    llm = ScriptedLLM(model="scripted-fake", replies={
        "Researcher": lambda m, n: final("(1) LangGraph models agents as graphs. (2) It checkpoints state. (3) It supports human-in-the-loop."),
        "Writer":     lambda m, n: final("LangGraph lets you build agents as graphs, saves their state, and can pause for a human."),
        "Reviewer":   lambda m, n: final("APPROVED: accurate and clear."),
    })
    researcher = Agent(role="Researcher", goal="Find 3 correct facts about {topic}",
                       backstory="A careful analyst who never guesses.", llm=llm, allow_delegation=False)
    writer     = Agent(role="Writer", goal="Explain {topic} to beginners",
                       backstory="A friendly technical writer.", llm=llm)
    reviewer   = Agent(role="Reviewer", goal="Check the text is correct and clear",
                       backstory="A strict editor.", llm=llm)

    research = Task(description="Research {topic}.", expected_output="3 numbered facts", agent=researcher)
    write    = Task(description="Write one paragraph about {topic}.", expected_output="one paragraph",
                    agent=writer, context=[research])               # explicit context
    review   = Task(description="Review the paragraph.", expected_output="APPROVED or REJECTED + reason",
                    agent=reviewer, context=[research, write])

    for a in [researcher, writer, reviewer]:
        print(f"Agent  role={a.role!r:13} goal={a.goal!r:42} llm={a.llm.model} delegation={a.allow_delegation} max_iter={a.max_iter}")
    for t in [research, write, review]:
        print(f"Task   {t.description!r:38} agent={t.agent.role:10} context={[c.agent.role for c in t.context] if isinstance(t.context, list) else 'not set'}")
    print("\nLLM calls so far:", len(llm.log))
else:
    print("skipped: pip install crewai")

Agent  role='Researcher'  goal='Find 3 correct facts about {topic}'       llm=scripted-fake delegation=False max_iter=25
Agent  role='Writer'      goal='Explain {topic} to beginners'             llm=scripted-fake delegation=False max_iter=25
Agent  role='Reviewer'    goal='Check the text is correct and clear'      llm=scripted-fake delegation=False max_iter=25
Task   'Research {topic}.'                    agent=Researcher context=not set
Task   'Write one paragraph about {topic}.'   agent=Writer     context=['Researcher']
Task   'Review the paragraph.'                agent=Reviewer   context=['Researcher', 'Writer']

LLM calls so far: 0


### 3.2 A sequential crew: `kickoff(inputs=...)`

`Crew.kickoff()` fills the placeholders, runs the tasks in order, and returns a `CrewOutput`: `.raw` (final text), `.tasks_output` (one `TaskOutput` per task), `.pydantic` / `.json_dict` (structured output, if requested) and `.token_usage`.

In [8]:
if HAVE_CREWAI:
    crew = Crew(agents=[researcher, writer, reviewer], tasks=[research, write, review],
                process=Process.sequential, verbose=False)
    result = kickoff(crew, inputs={"topic": "LangGraph"})

    print("Final (.raw):", result.raw)
    for t in result.tasks_output:
        print(f"  {t.agent:<10} -> {t.raw[:80]}")

    # Peek at the prompt CrewAI actually built for the Writer
    writer_call = next(c for c in llm.log if c["role"] == "Writer")
    print("\n--- system prompt sent to the Writer ---\n" + writer_call["messages"][0]["content"][:300])
    user_msg = writer_call["messages"][-1]["content"]
    start = user_msg.find("This is the context")
    print("--- user prompt, context part (shortened) ---\n" + user_msg[start:start + 230] + "\n   ...")
    print("Writer's prompt contains the Researcher's facts (context passing):", "checkpoints state" in user_msg)
else:
    print("skipped: pip install crewai")

Final (.raw): APPROVED: accurate and clear.
  Researcher -> (1) LangGraph models agents as graphs. (2) It checkpoints state. (3) It supports
  Writer     -> LangGraph lets you build agents as graphs, saves their state, and can pause for 
  Reviewer   -> APPROVED: accurate and clear.

--- system prompt sent to the Writer ---
You are Writer. A friendly technical writer.
Your personal goal is: Explain LangGraph to beginners
--- user prompt, context part (shortened) ---
This is the context you're working with:
(1) LangGraph models agents as graphs. (2) It checkpoints state. (3) It supports human-in-the-loop.

Provide your complete response:
   ...
Writer's prompt contains the Researcher's facts (context passing): True


### 3.3 Tools

**What it is.** A CrewAI tool is a Python function the agent may call. The simplest way is the `@tool("name")` decorator; the docstring becomes the description the LLM reads. For more control, subclass `BaseTool` with an `args_schema`. Ready-made tools (web search, file read, scraping) live in the separate `crewai-tools` package.

Below, our scripted Researcher first asks for the tool (call 1), then gives a final answer that uses the tool's result (call 2).

In [9]:
if HAVE_CREWAI:
    from crewai.tools import tool

    FAKE_INDEX = {"langgraph": "LangGraph: graph-based agent framework by LangChain (toy search result)",
                  "crewai": "CrewAI: role-based multi-agent framework (toy search result)"}
    tool_calls = []

    @tool("search_docs")
    def search_docs(query: str) -> str:
        """Search a small offline index of documentation snippets."""
        tool_calls.append(query)
        return FAKE_INDEX.get(query.lower(), "no result")

    def researcher_with_tool(messages, n):
        if n == 1:   # first call: ask for the tool, in CrewAI's text format
            return 'Thought: I should search first.\nAction: search_docs\nAction Input: {"query": "crewai"}'
        seen = " ".join(str(m.get("content", "")) for m in messages)      # the tool result is now in the prompt
        found = re.search(r"CrewAI: [^(]*", seen)
        return final(f"Based on the search: {found.group(0).strip() if found else 'nothing found'}.")

    tllm = ScriptedLLM(model="scripted-fake", replies={"Researcher": researcher_with_tool})
    tool_agent = Agent(role="Researcher", goal="Answer using the docs", backstory="Uses tools.",
                       llm=tllm, tools=[search_docs])
    tool_crew = Crew(agents=[tool_agent],
                     tasks=[Task(description="What is CrewAI?", expected_output="one sentence", agent=tool_agent)])
    out = kickoff(tool_crew)
    print("tool was called with:", tool_calls)
    print("LLM calls:", tllm.counts)
    print("answer:", out.raw)
else:
    print("skipped: pip install crewai")

tool was called with: ['crewai']
LLM calls: {'Researcher': 2}
answer: Based on the search: CrewAI: role-based multi-agent framework.


### 3.4 Structured output with Pydantic

**What it is.** Set `output_pydantic=MyModel` on a task, and CrewAI asks the LLM for JSON that matches the model's schema, then parses it. `result.pydantic` is a real Python object, so later code does not have to parse free text. (`output_json=` gives a plain dict instead.) For the general idea see [Structured outputs](../08_LLM/17_structured_outputs.ipynb).

In [10]:
if HAVE_CREWAI:
    from pydantic import BaseModel

    class Review(BaseModel):
        approved: bool
        score: int          # 1-10
        comments: str

    sllm = ScriptedLLM(model="scripted-fake", replies={
        "Reviewer": lambda m, n: final('{"approved": true, "score": 8, "comments": "Clear, add one example."}')})
    rev = Agent(role="Reviewer", goal="Review drafts", backstory="Strict editor.", llm=sllm)
    rtask = Task(description="Review this draft: 'LangGraph builds agents as graphs.'",
                 expected_output="a review", agent=rev, output_pydantic=Review)
    out = kickoff(Crew(agents=[rev], tasks=[rtask]))
    print(type(out.pydantic).__name__, "->", out.pydantic)
    print("approved?", out.pydantic.approved, "| score:", out.pydantic.score)
else:
    print("skipped: pip install crewai")

Review -> approved=True score=8 comments='Clear, add one example.'
approved? True | score: 8


### 3.5 Guardrails: check the output, retry with feedback

**What it is.** A **task guardrail** is a function that receives the task output and returns `(True, value)` to accept it or `(False, "feedback")` to reject it. On rejection CrewAI sends the feedback to the agent and retries, up to `guardrail_max_retries` times.

**Analogy.** A teacher who hands back homework with a note: "add a title, then resubmit".

CrewAI also accepts a plain-English guardrail (`guardrail="must be under 100 words"`), which uses an LLM to judge; a Python function is cheaper and deterministic, so prefer it when the rule can be coded.

In [11]:
if HAVE_CREWAI:
    def must_start_with_title(output):
        text = output.raw
        if text.startswith("TITLE:"):
            return (True, text)
        return (False, "Your answer must start with 'TITLE:'. Please fix it.")

    gllm = ScriptedLLM(model="scripted-fake", replies={
        "Writer": lambda m, n: final("LangGraph is great.") if n == 1 else final("TITLE: Why graphs\nLangGraph is great.")})
    gw = Agent(role="Writer", goal="Write posts", backstory="Writer.", llm=gllm)
    gtask = Task(description="Write a tiny post.", expected_output="a post with a title", agent=gw,
                 guardrail=must_start_with_title, guardrail_max_retries=2)
    out = kickoff(Crew(agents=[gw], tasks=[gtask]))
    print("writer was called", gllm.counts["Writer"], "times (1st failed the guardrail)")
    print("accepted output:", out.raw.replace("\n", " | "))
else:
    print("skipped: pip install crewai")

writer was called 2 times (1st failed the guardrail)
accepted output: TITLE: Why graphs | LangGraph is great.


### 3.6 Hierarchical process: a manager delegates

**What it is.** With `process=Process.hierarchical`, you give a `manager_llm` (or your own `manager_agent`). CrewAI creates a **"Crew Manager"** agent that has two built-in tools: **`delegate_work_to_coworker`** and **`ask_question_to_coworker`**. Tasks do not need a fixed agent; the manager decides who does what, then reviews and combines the results.

Our scripted manager delegates research, then writing, then answers. Watch the order of LLM calls.

In [12]:
if HAVE_CREWAI:
    def manager_script(messages, n):
        if n == 1:
            return ('Thought: First get facts.\nAction: delegate_work_to_coworker\n'
                    'Action Input: {"task": "Find 2 facts about CrewAI", "context": "for a beginner blog", "coworker": "Researcher"}')
        if n == 2:
            return ('Thought: Now get it written.\nAction: delegate_work_to_coworker\n'
                    'Action Input: {"task": "Write 1 sentence about CrewAI", "context": "facts: role-based; has Flows", "coworker": "Writer"}')
        return final("CrewAI is a role-based multi-agent framework that also offers event-driven Flows.")

    hllm = ScriptedLLM(model="scripted-fake", replies={
        "Crew Manager": manager_script,
        "Researcher": lambda m, n: final("(1) CrewAI is role-based. (2) It has Flows."),
        "Writer":     lambda m, n: final("CrewAI organises agents by role and also offers Flows.")})
    h_res = Agent(role="Researcher", goal="find facts", backstory="analyst", llm=hllm)
    h_wri = Agent(role="Writer", goal="write", backstory="writer", llm=hllm)
    hcrew = Crew(agents=[h_res, h_wri],
                 tasks=[Task(description="Produce a one-sentence intro to {topic}.", expected_output="one sentence")],
                 process=Process.hierarchical, manager_llm=hllm)
    out = kickoff(hcrew, inputs={"topic": "CrewAI"})
    print("order of LLM calls:", " -> ".join(c["role"] for c in hllm.log))
    print("final:", out.raw)
else:
    print("skipped: pip install crewai")

order of LLM calls: Crew Manager -> Researcher -> Crew Manager -> Writer -> Crew Manager
final: CrewAI is a role-based multi-agent framework that also offers event-driven Flows.


**Sequential vs hierarchical**

| | Sequential | Hierarchical |
|---|---|---|
| Who picks the agent | you (each task has `agent=`) | the manager LLM |
| LLM calls | about one loop per task | extra manager calls for every delegation and review |
| Predictability | high | lower (the manager may delegate differently each run) |
| Good for | known pipelines | open-ended work where the split is not known in advance |

### 3.7 `kickoff_async` and `kickoff_for_each`

- `await crew.kickoff_async(inputs=...)` runs a crew without blocking, so several crews can run concurrently (inside `asyncio`).
- `crew.kickoff_for_each(inputs=[{...}, {...}])` runs the same crew once per input (a simple batch). There is also `kickoff_for_each_async`.

In [13]:
if HAVE_CREWAI:
    def brief_script(m, n):
        topic = re.search(r"Summarise (\w+)", m[-1]["content"]).group(1)
        return final(f"{topic}: a toy one-line summary.")
    bllm = ScriptedLLM(model="scripted-fake", replies={"Summariser": brief_script})
    summariser = Agent(role="Summariser", goal="summarise", backstory="concise", llm=bllm)
    make_crew = lambda: Crew(agents=[summariser],
                             tasks=[Task(description="Summarise {topic} in one line.", expected_output="one line", agent=summariser)])

    with ThreadPoolExecutor(1) as pool:      # same Jupyter detail as kickoff()
        batch = pool.submit(make_crew().kickoff_for_each, inputs=[{"topic": "LangGraph"}, {"topic": "CrewAI"}]).result()
    print("kickoff_for_each:", [r.raw for r in batch])

    async def run_two():
        return await asyncio.gather(make_crew().kickoff_async(inputs={"topic": "AutoGen"}),
                                    make_crew().kickoff_async(inputs={"topic": "smolagents"}))
    # In a plain script you would write: results = asyncio.run(run_two())
    # Jupyter already runs an event loop, so here we run it in a helper thread (or use `await run_two()`).
    with ThreadPoolExecutor(1) as pool:
        results = pool.submit(asyncio.run, run_two()).result()
    print("kickoff_async   :", [r.raw for r in results])
else:
    print("skipped: pip install crewai")

kickoff_for_each: ['LangGraph: a toy one-line summary.', 'CrewAI: a toy one-line summary.']


kickoff_async   : ['AutoGen: a toy one-line summary.', 'smolagents: a toy one-line summary.']


### 3.8 Flows: event-driven workflows with typed state

**What it is.** A CrewAI **Flow** is the "real" version of our mini Flow (section 2.1). It is the recommended way to build **production** apps in CrewAI: plain Python steps, LLM calls and whole crews glued together by events, with a typed, persistent `state`.

| Decorator | Meaning |
|---|---|
| `@start()` | entry point(s) of the flow |
| `@listen(method_or_label)` | run after that method finishes (or after a router returns that label); receives its return value |
| `@router(method)` | run after `method`, return a label string to choose the branch |
| `or_(a, b)` / `and_(a, b)` | run when **any** / **all** of the triggers have fired |
| `@persist` | save the state (default SQLite) so a flow can resume later |

Typed state: subclass `Flow[MyState]` where `MyState` is a Pydantic model. Below, a router sends the draft back for a rewrite until it scores high enough, which is a **loop** in an event-driven style.

In [14]:
if HAVE_CREWAI:
    from crewai.flow.flow import Flow, start, listen, router, or_
    from pydantic import BaseModel

    class BlogState(BaseModel):
        topic: str = ""
        draft: str = ""
        score: int = 0
        attempts: int = 0

    class BlogFlow(Flow[BlogState]):
        @start()
        def pick_topic(self):
            self.state.topic = "agent frameworks"

        @listen(or_(pick_topic, "retry"))          # runs first, and again on every "retry"
        def write_draft(self):
            self.state.attempts += 1
            # In a real flow this step could be: MyCrew().crew().kickoff(inputs={...})
            self.state.draft = f"Draft v{self.state.attempts} on {self.state.topic}"
            self.state.score = 5 if self.state.attempts == 1 else 9      # scripted scores

        @router(write_draft)
        def check(self):
            if self.state.attempts >= 3: return "give_up"                 # safety stop
            return "good" if self.state.score >= 7 else "retry"

        @listen("good")
        def publish(self):
            return f"PUBLISHED: {self.state.draft} (score {self.state.score})"

        @listen("give_up")
        def escalate(self):
            return "Sent to a human editor."

    blog = BlogFlow(suppress_flow_events=True)       # hide the console panels
    print(blog.kickoff())
    print("state:", blog.state.model_dump(exclude={"id"}))
else:
    print("skipped: pip install crewai")

PUBLISHED: Draft v2 on agent frameworks (score 9)
state: {'topic': 'agent frameworks', 'draft': 'Draft v2 on agent frameworks', 'score': 9, 'attempts': 2}


**Crews vs Flows.** A **Crew** is "agents collaborating with some autonomy". A **Flow** is "precise, event-driven control". The common production pattern is a Flow that calls one or more Crews for the fuzzy parts, and plain code for everything else.

### 3.9 Memory and knowledge (not run here: they need an embedding model)

- **Memory**: `Crew(..., memory=True)` lets agents remember facts across tasks and runs. Recent 1.x versions use one unified `Memory` class with scopes; older versions split it into short-term, long-term and entity memory. Memory needs an **embedder** (a model that turns text into vectors, by default a hosted one), so it does not run offline here. You can pass `embedder={...}` to use a local model.
- **Knowledge**: documents the agents can search (like built-in RAG). Example: `knowledge_sources=[StringKnowledgeSource(content="Our refund window is 30 days.")]` on an `Agent` or `Crew`; file sources exist for text, PDF, CSV, Excel and JSON.

```python
# Sketch only (needs an embedder / API key)
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource
policy = StringKnowledgeSource(content="Refunds are allowed within 30 days of purchase.")
support = Agent(role="Support", goal="Answer policy questions", backstory="...",
                knowledge_sources=[policy])
crew = Crew(agents=[support], tasks=[...], memory=True)
```
For memory design in general (what to store, when to forget) see [notebook 11](11_context_engineering_memory_and_agent_protocols.ipynb).

### 3.10 The YAML project layout (`crewai create crew`)

The CLI command `crewai create crew my_project` scaffolds a project where agents and tasks live in **YAML** files (easy for non-programmers to edit) and the wiring lives in Python:

```
my_project/
├── .env                      # API keys (never commit this)
├── pyproject.toml
├── knowledge/                # files for knowledge sources
└── src/my_project/
    ├── main.py               # calls MyProjectCrew().crew().kickoff(inputs=...)
    ├── crew.py               # @CrewBase class with @agent / @task / @crew methods
    ├── tools/custom_tool.py
    └── config/
        ├── agents.yaml
        └── tasks.yaml
```

```yaml
# config/agents.yaml
researcher:
  role: "{topic} Researcher"
  goal: "Find 3 correct facts about {topic}"
  backstory: "A careful analyst who never guesses."

# config/tasks.yaml
research_task:
  description: "Research {topic}."
  expected_output: "3 numbered facts"
  agent: researcher
```

```python
# crew.py
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task

@CrewBase
class MyProjectCrew:
    agents_config = "config/agents.yaml"
    tasks_config = "config/tasks.yaml"

    @agent
    def researcher(self) -> Agent:
        return Agent(config=self.agents_config["researcher"])

    @task
    def research_task(self) -> Task:
        return Task(config=self.tasks_config["research_task"])

    @crew
    def crew(self) -> Crew:
        return Crew(agents=self.agents, tasks=self.tasks, process=Process.sequential)
```

Run it with `crewai run` (or `python -m my_project.main`). `crewai create flow my_flow` scaffolds a Flow project in the same style.

In [15]:
if HAVE_CREWAI:
    from crewai.project import CrewBase, agent, task, crew   # the decorators used in crew.py
    print("crewai.project decorators available:", [f.__name__ for f in (agent, task, crew)], "+ CrewBase")
else:
    print("skipped: pip install crewai")

crewai.project decorators available: ['agent', 'task', 'crew'] + CrewBase


## 4. One pipeline, many frameworks: research -> write -> review

To compare frameworks fairly we build the **same** 3-agent pipeline in each: a **researcher** finds facts, a **writer** drafts a paragraph, a **reviewer** approves or asks for a revision (at most 2 revisions).

```mermaid
flowchart LR
    S([topic]) --> R[researcher] --> W[writer] --> V{reviewer}
    V -- revise --> W
    V -- approved --> E([final text])
```

### 4.1 LangGraph (runnable)

The agents are scripted stand-ins; the graph, state and loop are real LangGraph.

In [16]:
from langgraph.graph import StateGraph, START, END

class PipeState(TypedDict):
    topic: str
    facts: str
    draft: str
    feedback: Annotated[list, operator.add]
    revisions: int

def lg_researcher(s):   # stand-in for an LLM + search tool
    return {"facts": "(1) it uses graphs (2) it saves checkpoints"}

def lg_writer(s):
    extra = " For example, it can pause for a human." if s["feedback"] else ""
    facts = s["facts"].replace("(1) ", "").replace(" (2) ", " and ")
    return {"draft": f"{s['topic']} is an agent library: {facts}.{extra}",
            "revisions": s["revisions"] + (1 if s["feedback"] else 0)}

def lg_reviewer(s):
    ok = "example" in s["draft"].lower()                         # scripted review rule
    return {"feedback": ["APPROVED" if ok else "add an example"]}

def after_review(s) -> str:
    if s["feedback"][-1] == "APPROVED" or s["revisions"] >= 2:   # quality stop + safety stop
        return "done"
    return "revise"

g = StateGraph(PipeState)
g.add_node("researcher", lg_researcher); g.add_node("writer", lg_writer); g.add_node("reviewer", lg_reviewer)
g.add_edge(START, "researcher"); g.add_edge("researcher", "writer"); g.add_edge("writer", "reviewer")
g.add_conditional_edges("reviewer", after_review, {"revise": "writer", "done": END})
pipeline = g.compile()

for step in pipeline.stream({"topic": "LangGraph", "feedback": [], "revisions": 0}, stream_mode="updates"):
    print(" ", step)

  {'researcher': {'facts': '(1) it uses graphs (2) it saves checkpoints'}}
  {'writer': {'draft': 'LangGraph is an agent library: it uses graphs and it saves checkpoints.', 'revisions': 0}}
  {'reviewer': {'feedback': ['add an example']}}
  {'writer': {'draft': 'LangGraph is an agent library: it uses graphs and it saves checkpoints. For example, it can pause for a human.', 'revisions': 1}}
  {'reviewer': {'feedback': ['APPROVED']}}


### 4.2 CrewAI (runnable if installed)

In CrewAI the "revise until approved" loop is not a crew feature: you either use a **guardrail** on the writing task (the reviewer's rule becomes the guardrail), or a **Flow** with a router (section 3.8). Here we use a guardrail, plus a separate review task.

In [17]:
if HAVE_CREWAI:
    pllm = ScriptedLLM(model="scripted-fake", replies={
        "Researcher": lambda m, n: final("(1) CrewAI is role-based (2) it has Flows"),
        "Writer":     lambda m, n: final("CrewAI is a role-based framework." if n == 1
                                         else "CrewAI is a role-based framework. For example, a crew can research and write."),
        "Reviewer":   lambda m, n: final("APPROVED"),
    })
    mk = lambda role: Agent(role=role, goal=f"act as {role}", backstory="team member", llm=pllm)
    r_, w_, v_ = mk("Researcher"), mk("Writer"), mk("Reviewer")
    def has_example(out):
        return (True, out.raw) if "example" in out.raw.lower() else (False, "add an example")
    t_research = Task(description="Research {topic}.", expected_output="facts", agent=r_)
    t_write = Task(description="Write a paragraph on {topic}.", expected_output="paragraph", agent=w_,
                   guardrail=has_example, guardrail_max_retries=2)
    t_review = Task(description="Review the paragraph.", expected_output="APPROVED or feedback", agent=v_)
    res = kickoff(Crew(agents=[r_, w_, v_], tasks=[t_research, t_write, t_review]), inputs={"topic": "CrewAI"})
    for t in res.tasks_output:
        print(f"  {t.agent:<10} -> {t.raw}")
    print("  writer calls:", pllm.counts["Writer"])
else:
    print("skipped: pip install crewai")

  Researcher -> (1) CrewAI is role-based (2) it has Flows
  Writer     -> CrewAI is a role-based framework. For example, a crew can research and write.
  Reviewer   -> APPROVED
  writer calls: 2


### 4.3 The same pipeline in other frameworks (text only, version-agnostic)

These snippets are **sketches** to show each framework's *shape*. They are not run (they need the package and an API key), and exact class names change between releases, so check the current docs. `"<model>"` stands for whichever model you use.

**AutoGen (AgentChat API, v0.4+) / Microsoft Agent Framework.** Conversation-centred: agents take turns in a **team** (group chat) until a termination condition fires.
```python
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

model = OpenAIChatCompletionClient(model="<model>")
researcher = AssistantAgent("researcher", model_client=model, system_message="Find 3 facts about the topic.")
writer     = AssistantAgent("writer", model_client=model, system_message="Write one paragraph from the facts.")
reviewer   = AssistantAgent("reviewer", model_client=model, system_message="Review. Reply APPROVE when it is good.")
team = RoundRobinGroupChat([researcher, writer, reviewer],
                           termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(9))
result = asyncio.run(team.run(task="Topic: LangGraph"))
```
Microsoft has announced **Microsoft Agent Framework** as the successor that merges AutoGen and Semantic Kernel. Its shape is similar: create agents from a chat client with `instructions=`, then compose them with workflow builders (for example a sequential, concurrent, group-chat or handoff builder). Names were still settling during its preview, so follow its docs.

**OpenAI Agents SDK.** Lightweight: `Agent` + `Runner`, with **handoffs** (pass control to another agent) and **agents-as-tools** (a manager calls specialists like functions), plus built-in tracing and guardrails.
```python
from agents import Agent, Runner

researcher = Agent(name="Researcher", instructions="Find 3 facts about the topic.")
writer     = Agent(name="Writer", instructions="Write one paragraph from the facts.")
reviewer   = Agent(name="Reviewer", instructions="Reply APPROVED or give one fix.")

facts  = Runner.run_sync(researcher, "Topic: LangGraph").final_output     # plain code orchestration
draft  = Runner.run_sync(writer, f"Facts:\n{facts}").final_output
review = Runner.run_sync(reviewer, f"Draft:\n{draft}").final_output

# Alternative: a manager that uses the others as tools
manager = Agent(name="Editor-in-chief", instructions="Research, write, then review.",
                tools=[researcher.as_tool(tool_name="research", tool_description="find facts"),
                       writer.as_tool(tool_name="write", tool_description="write a draft"),
                       reviewer.as_tool(tool_name="review", tool_description="review a draft")])
```

**Google Agent Development Kit (ADK).** Composable agent classes: `LlmAgent` for LLM steps and **workflow agents** (`SequentialAgent`, `ParallelAgent`, `LoopAgent`) for structure. Agents share a session **state**; `output_key` saves an agent's answer into it and `{key}` reads it in instructions.
```python
from google.adk.agents import LlmAgent, SequentialAgent

researcher = LlmAgent(name="researcher", model="<model>", instruction="Find 3 facts about {topic}.", output_key="facts")
writer     = LlmAgent(name="writer", model="<model>", instruction="Write a paragraph using: {facts}", output_key="draft")
reviewer   = LlmAgent(name="reviewer", model="<model>", instruction="Review this draft: {draft}", output_key="review")
pipeline   = SequentialAgent(name="blog_pipeline", sub_agents=[researcher, writer, reviewer])
# Run with an ADK Runner + session service (e.g. an in-memory runner), or the `adk run` / `adk web` CLI.
# A LoopAgent around writer + reviewer gives the "revise until approved" loop.
```

**PydanticAI.** Type-safe agents from the Pydantic team. Each agent can return a validated Pydantic model; multi-agent is usually plain Python calling agents in order, or one agent calling another inside a tool.
```python
from pydantic import BaseModel
from pydantic_ai import Agent

class Review(BaseModel):
    approved: bool
    comments: str

researcher = Agent("<provider:model>", system_prompt="Find 3 facts about the topic.")
writer     = Agent("<provider:model>", system_prompt="Write one paragraph from the facts.")
reviewer   = Agent("<provider:model>", output_type=Review, system_prompt="Review the draft.")

facts  = researcher.run_sync("Topic: LangGraph").output
draft  = writer.run_sync(f"Facts: {facts}").output
review = reviewer.run_sync(f"Draft: {draft}").output        # a Review object, already validated
```

**smolagents (Hugging Face).** Minimal and code-first: a `CodeAgent` writes and runs Python to call tools; a manager agent can **manage** other agents, which it calls like tools.
```python
from smolagents import CodeAgent, ToolCallingAgent, InferenceClientModel

model = InferenceClientModel()            # or another model class, e.g. a LiteLLM-based one
researcher = ToolCallingAgent(tools=[], model=model, name="researcher", description="Finds facts about a topic.")
writer     = ToolCallingAgent(tools=[], model=model, name="writer", description="Writes a paragraph from facts.")
reviewer   = ToolCallingAgent(tools=[], model=model, name="reviewer", description="Reviews a draft, says APPROVED or a fix.")
manager = CodeAgent(tools=[], model=model, managed_agents=[researcher, writer, reviewer])
manager.run("Research LangGraph, write a paragraph, get it reviewed, revise once if needed.")
```

**LlamaIndex Workflows.** Event-driven steps: each `@step` method receives one event type and returns the next event. (LlamaIndex also has `AgentWorkflow` for agent handoffs.)
```python
from llama_index.core.workflow import Workflow, StartEvent, StopEvent, Event, step

class Facts(Event):  facts: str
class Draft(Event):  draft: str

class BlogPipeline(Workflow):
    @step
    async def research(self, ev: StartEvent) -> Facts:
        return Facts(facts=await call_llm(f"3 facts about {ev.topic}"))      # call_llm = your LLM call
    @step
    async def write(self, ev: Facts) -> Draft:
        return Draft(draft=await call_llm(f"Paragraph from: {ev.facts}"))
    @step
    async def review(self, ev: Draft) -> StopEvent:
        return StopEvent(result=await call_llm(f"Review: {ev.draft}"))

result = await BlogPipeline(timeout=60).run(topic="LangGraph")
```

## 5. Comparison table

A rough, qualitative guide as of late 2026. All of these move quickly; treat it as a starting point, not a ranking.

| Framework | Mental model | Control vs ease | State | Memory | Human-in-the-loop | Observability | Best for |
|---|---|---|---|---|---|---|---|
| **LangGraph** | explicit graph of nodes and edges | most control, more code | typed shared state + reducers, checkpoints | checkpointer (thread) + Store (long-term) | first-class: `interrupt()`, resume, time travel | LangSmith; OpenTelemetry-based tools | complex, long-running, stateful agents in production |
| **CrewAI** | crew of role-playing agents + Flows | very easy to start; Flows add control | task outputs as context; typed Flow state | built-in memory and knowledge (needs embedder) | human input on tasks; Flow steps can wait for people | built-in event bus, integrations with tracing tools, a commercial platform | role-based pipelines, quick prototypes, business workflows |
| **AutoGen / MS Agent Framework** | agents chatting in teams; workflows | easy for chat-style collaboration | conversation history (+ workflow state in MAF) | pluggable memory components | user-proxy agents / approval steps | OpenTelemetry support | research-style conversations, code-executing agents, Microsoft/Azure stacks |
| **OpenAI Agents SDK** | agents + handoffs + agents-as-tools | very easy, small API | conversation + sessions | sessions | approval hooks on tools | built-in tracing dashboard | quick production agents, especially with OpenAI models |
| **Google ADK** | LLM agents + workflow agents (sequential, parallel, loop) | easy, structured | shared session state (`output_key`) | session + memory services | tool confirmation / callbacks | dev UI, Cloud tracing | Gemini / Google Cloud deployments, multi-agent hierarchies |
| **PydanticAI** | type-safe agent + plain Python | easy, very explicit | your own Python objects; optional graph module | your choice | deferred / approval-required tools | Logfire (OpenTelemetry) | production apps that need validated, typed outputs |
| **smolagents** | code-writing agents | minimal, very easy | agent memory of steps | simple step memory | limited, DIY | OpenTelemetry instrumentation available | small agents, open models, experiments |
| **LlamaIndex Workflows** | event-driven steps | medium | context object shared by steps | LlamaIndex memory modules | wait-for-event steps | many integrations | RAG-heavy and document agents |

**Rules of thumb**
- Need loops, pausing, resuming and exact control: **LangGraph** (or PydanticAI / ADK with workflow agents).
- Want a readable team of roles quickly: **CrewAI** (move hot paths into Flows later).
- Mostly one vendor's models and want the least code: that vendor's SDK (**OpenAI Agents SDK**, **Google ADK**).
- Typed outputs and plain Python are what you care about most: **PydanticAI**.
- Your framework matters less than your **tools, prompts, evaluations and harness** (see [harness engineering](10_harness_engineering_and_agentic_engineering.ipynb)).

## 6. How multi-agent systems fail (and small defences you can code)

Several agents means several LLM calls, several prompts and several hand-offs, and each is a new place for things to go wrong. Research that studied many multi-agent traces (Cemri et al., "Why Do Multi-Agent LLM Systems Fail?", 2025) grouped failures into three families: **bad specification / design**, **misalignment between agents**, and **weak verification / termination**.

| Failure mode | What it looks like | Defence |
|---|---|---|
| **Coordination loop** | agents keep passing work back and forth ("you do it" / "no, you") | max turns, loop detection on repeated states, a clear owner per task |
| **Cost blow-up** | every agent re-reads the whole history; managers add extra calls | budgets per run, short hand-off messages, fewer agents, cheaper models for workers |
| **Error cascade** | one agent's mistake is trusted and amplified by the next | validate at each hand-off (schemas, fact checks against sources), independent reviewers |
| **Context loss at hand-off** | the next agent misses a key detail | structured hand-off notes, shared state instead of paraphrase |
| **Role drift / confusion** | the writer starts doing research, two agents do the same job | narrow roles, explicit task outputs |
| **Premature or no termination** | stops before finishing, or never says "done" | explicit "definition of done", stop rules in code, not only in prompts |
| **Sycophantic review** | the reviewer approves everything | give the reviewer concrete checks, or use code checks instead |

### 6.1 Coordination loop + loop detector

Two scripted agents each think the other should do the task. Without a guard they would bounce forever; a **max turn** limit and a **repeated-state detector** stop them.

In [18]:
def agent_a(msg): return "B should handle this"      # scripted stand-ins
def agent_b(msg): return "A should handle this"

def run_with_guards(max_turns=10):
    seen, msg, speaker = set(), "please do task X", "A"
    for turn in range(1, max_turns + 1):
        msg = agent_a(msg) if speaker == "A" else agent_b(msg)
        state = (speaker, msg)
        if state in seen:                                # same speaker said the same thing before
            return f"loop detected at turn {turn}: {state} -> escalate to a human / supervisor"
        seen.add(state)
        speaker = "B" if speaker == "A" else "A"
    return "stopped: max turns reached"

print(run_with_guards())

loop detected at turn 3: ('A', 'B should handle this') -> escalate to a human / supervisor


### 6.2 Cost blow-up: count the calls and tokens

A toy cost model (illustrative numbers only, not real prices): each LLM call costs `input_tokens + output_tokens`, and each agent re-reads the conversation so far. We compare a single agent with a supervisor setup and with a "group chat" where every agent sees every message.

In [19]:
def simulate(pattern: str, steps: int = 3, msg_tokens: int = 300, system_tokens: int = 500):
    """Return (llm_calls, total_tokens) for a toy run. All numbers are illustrative."""
    history, calls, tokens = 0, 0, 0
    def call(extra_readers=1):
        nonlocal history, calls, tokens
        for _ in range(extra_readers):
            calls += 1
            tokens += system_tokens + history + msg_tokens     # read history, write a message
        history += msg_tokens * extra_readers
    for _ in range(steps):
        if pattern == "single agent":
            call()
        elif pattern == "supervisor + workers":
            call(); call()                                       # supervisor decides, worker acts
        elif pattern == "group chat (4 agents)":
            call(extra_readers=4)                                # everyone reacts to everything
    return calls, tokens

print(f"{'pattern':<24}{'LLM calls':>10}{'tokens':>10}")
for p in ["single agent", "supervisor + workers", "group chat (4 agents)"]:
    c, t = simulate(p)
    print(f"{p:<24}{c:>10}{t:>10}")

class Budget:
    """A simple guard: stop the run when the token budget is used up."""
    def __init__(self, max_tokens): self.max, self.used = max_tokens, 0
    def charge(self, n):
        self.used += n
        if self.used > self.max:
            raise RuntimeError(f"budget exceeded: {self.used} > {self.max} tokens")
b = Budget(5_000)
try:
    for i in range(20): b.charge(800)
except RuntimeError as e:
    print("\nBudget guard:", e, f"(after {i + 1} calls)")

pattern                  LLM calls    tokens
single agent                     3      3300
supervisor + workers             6      9300
group chat (4 agents)           12     24000

Budget guard: budget exceeded: 5600 > 5000 tokens (after 7 calls)


The same work cost several times more tokens once more agents re-read a growing history. Anthropic reported that their multi-agent research system used many times more tokens than a single chat ("How we built our multi-agent research system", 2025), which pays off only for high-value, highly parallel tasks.

### 6.3 Error cascade + a validation gate

The scripted researcher **hallucinates** (makes up) a fact. The writer trusts it, and a style-only reviewer approves. A **validation gate** that checks claims against the source documents stops the cascade at the first hand-off.

In [20]:
SOURCES = ["LangGraph was released by LangChain.", "LangGraph supports checkpointing."]

def researcher_bad(topic):              # scripted: one true fact, one invented fact
    return ["LangGraph supports checkpointing.", "LangGraph was invented in 1995."]

def writer_trusting(facts):  return "Fun facts: " + " ".join(facts)
def reviewer_style_only(text): return "APPROVED" if text.startswith("Fun facts") else "REJECTED"

def validate(facts):
    """Gate: keep only claims that appear in a source (a stand-in for real fact checking)."""
    good = [f for f in facts if any(f.lower() in s.lower() for s in SOURCES)]
    bad = [f for f in facts if f not in good]
    return good, bad

facts = researcher_bad("LangGraph")
text = writer_trusting(facts)
print("WITHOUT gate:", reviewer_style_only(text), "->", text)

good, bad = validate(facts)
print("WITH gate   : rejected", bad)
print("              ", reviewer_style_only(writer_trusting(good)), "->", writer_trusting(good))

WITHOUT gate: APPROVED -> Fun facts: LangGraph supports checkpointing. LangGraph was invented in 1995.
WITH gate   : rejected ['LangGraph was invented in 1995.']
               APPROVED -> Fun facts: LangGraph supports checkpointing.


## 7. When NOT to use multi-agent

**Default to one agent.** A single agent with good tools and a good prompt is cheaper, faster, easier to debug and often just as accurate. Anthropic's "Building effective agents" (2024) and Cognition's "Don't Build Multi-Agents" (2025) both argue for starting simple; the main reason is that **context gets split** between agents, so each one decides with only part of the picture.

```mermaid
flowchart TD
    A[New task] --> B{Can one prompt + tools do it?}
    B -- yes --> S[Single agent]
    B -- no --> C{Is the flow known in advance?}
    C -- yes --> W[Workflow / pipeline<br/>fixed steps in code or a graph]
    C -- no --> D{Are there independent sub-tasks<br/>that can run in parallel?}
    D -- yes --> M[Multi-agent: orchestrator + parallel workers]
    D -- no --> E{Do sub-tasks need very different<br/>tools, permissions or context?}
    E -- yes --> M
    E -- no --> S
```

| Use multi-agent when... | Avoid it when... |
|---|---|
| sub-tasks are truly **independent** and can run in parallel (broad research) | steps depend tightly on each other (most coding tasks) |
| one agent's context window would overflow | the task fits comfortably in one context |
| roles need **different tools or permissions** (security separation) | the "roles" are just different prompts for the same job |
| the task is valuable enough to pay several times the tokens | cost or latency matters a lot |
| you have evaluations to prove it beats a single agent | you have not measured a single-agent baseline yet |

## 8. Recap / cheat sheet

| Idea | Remember this |
|---|---|
| Agent (CrewAI) | `role` + `goal` + `backstory` become the system prompt; add `tools` and `llm` |
| Task | `description` + `expected_output`; `context=[...]` passes earlier outputs (default in sequential: all earlier outputs) |
| Crew | `agents` + `tasks` + `process`; `kickoff(inputs=...)` fills `{placeholders}` |
| Sequential vs hierarchical | fixed order vs a manager that delegates with tools (more calls, less predictable) |
| Tools | `@tool("name")` with a docstring; text ReAct format when native function calling is off |
| Structured output | `output_pydantic=Model` -> `result.pydantic` |
| Guardrail | `(output) -> (ok, value_or_feedback)`, automatic retries |
| Flow | `@start`, `@listen`, `@router`, `or_`/`and_`, typed `Flow[State]`, `@persist` |
| Offline testing | subclass `BaseLLM` with scripted replies |
| Framework choice | control (LangGraph) vs ease (CrewAI, vendor SDKs) vs types (PydanticAI) |
| Failure modes | loops, cost blow-ups, error cascades; defend with limits, budgets and validation gates |
| Default | one good agent first; multi-agent only with a reason and an evaluation |

## 9. Interview questions

1. **What are the three core CrewAI objects?** `Agent` (role, goal, backstory, tools, llm), `Task` (description, expected_output, agent, context) and `Crew` (agents, tasks, process).
2. **How does context pass between tasks?** The outputs of the tasks listed in `context` are added to the next task's prompt; in a sequential crew with no `context` set, a task sees all earlier outputs.
3. **Sequential vs hierarchical process?** Sequential runs tasks in a fixed order with fixed agents. Hierarchical adds a manager LLM that delegates with `delegate_work_to_coworker` / `ask_question_to_coworker` tools and reviews results; more flexible, more expensive, less predictable.
4. **Crew vs Flow?** A Crew gives agents autonomy to collaborate; a Flow is event-driven code (`@start/@listen/@router`) with typed state, used for precise control and production apps, often calling crews inside steps.
5. **How do you test a CrewAI or LangGraph app without paying for LLM calls?** Inject a fake LLM (a `BaseLLM` subclass or a fake chat model) with scripted replies, and assert on the control flow, tool calls and outputs.
6. **When would you choose LangGraph over CrewAI?** When you need explicit loops and branches, checkpoints, pause/resume, time travel or fine control over state; CrewAI when a role-based pipeline is enough and speed of building matters.
7. **Name three multi-agent failure modes and a defence for each.** Coordination loops (turn limits, loop detection), cost blow-ups (budgets, fewer agents, short hand-offs), error cascades (validation gates, independent checks).
8. **When should you not use multi-agent?** When one agent with tools can do it, when steps are tightly coupled, when cost/latency matter, or when you have not shown with evaluations that it beats a single-agent baseline.
9. **What is the role of a guardrail on a task?** To check the output with code (or an LLM) and automatically retry with feedback until it passes or the retry limit is hit.
10. **Agents-as-tools vs handoffs?** With agents-as-tools a manager stays in charge and calls specialists like functions; with handoffs control moves to the other agent, which then talks to the user directly.

## 10. Further reading

- CrewAI documentation (concepts: agents, tasks, crews, flows, tools, memory, knowledge, guardrails, custom LLMs) - https://docs.crewai.com/
- LangGraph documentation, multi-agent concepts - https://langchain-ai.github.io/langgraph/ ; and this repo's [LangGraph hands-on](12_langgraph_hands_on.ipynb)
- AutoGen documentation - https://microsoft.github.io/autogen/ ; Wu et al., "AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation" (2023), arXiv:2308.08155
- Microsoft Agent Framework documentation (Microsoft Learn) and GitHub repository `microsoft/agent-framework`
- OpenAI Agents SDK documentation - https://openai.github.io/openai-agents-python/
- Google Agent Development Kit documentation - https://google.github.io/adk-docs/
- PydanticAI documentation - https://ai.pydantic.dev/
- smolagents documentation - https://huggingface.co/docs/smolagents/
- LlamaIndex Workflows documentation (docs.llamaindex.ai)
- Cemri et al., "Why Do Multi-Agent LLM Systems Fail?" (2025), arXiv:2503.13657
- Hong et al., "MetaGPT: Meta Programming for a Multi-Agent Collaborative Framework" (2023), arXiv:2308.00352
- Anthropic, "Building effective agents" (2024) and "How we built our multi-agent research system" (2025), Anthropic engineering blog
- Cognition, "Don't Build Multi-Agents" (2025), Cognition blog
- In this repo: [04_multi_agent_frameworks](04_multi_agent_frameworks.ipynb), [07_advanced_agent_frameworks](07_advanced_agent_frameworks.ipynb), [10_harness_engineering_and_agentic_engineering](10_harness_engineering_and_agentic_engineering.ipynb), [11_context_engineering_memory_and_agent_protocols](11_context_engineering_memory_and_agent_protocols.ipynb)